# DPSynth Quickstart Demo

This notebook demonstrates how to use DPSynth to generate Differentially Private synthetic tabular data using the In-Memory DataFrame API.

In [ ]:
#@title Install DPSynth
!pip install git+https://github.com/google/dpsynth.git

## Download Dataset

To run this notebook with a realistic dataset, we will use the Adult Income dataset from Kaggle. You will need a Kaggle account and an API token (`kaggle.json`) to download it.

In [ ]:
# Install Kaggle API
!pip install kagglehub

import kagglehub

kagglehub.login()

# If you are using Colab, you can alternatively set KAGGLE_USERNAME and KAGGLE_KEY
# values in user data, and then uncomment and run the following code:
#
# from colabtools import userdata
#
# os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
# os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')
#
# You use userdata to keep the Kaggle API key safe. Alternatively, you can
# hardcode the values but it is not recommended due to security risks of
# leaking the API key.


# If you're not using Colab, set the env vars as appropriate for your system.
# For example, to set the env vars on Linux you can run in terminal:
# ```
# export KAGGLE_USERNAME="your_username"
# export KAGGLE_KEY="your_key"
# ```

# Download latest version
ds_path = kagglehub.dataset_download("wenruliu/adult-income-dataset")

print("Path to dataset files:", ds_path)

In [ ]:
import os
import dpsynth
from dpsynth import discrete_mechanisms
from dpsynth import domain
import pandas as pd

# 1. Load the downloaded dataset
sensitive_df = pd.read_csv(os.path.join(ds_path, 'adult.csv'))

print("Original Sensitive Data (sample):")
print(sensitive_df[['age', 'education']].head())

# 2. Define domain schema
# We consider age as Categorical to avoid discretization issues as suggested by reviewer.
attribute_domains = {
    'age': domain.CategoricalAttribute(possible_values=list(range(17, 91))), # Adult dataset age range
    'education': domain.CategoricalAttribute(possible_values=sensitive_df['education'].unique().tolist())
}

# 3. Configure the synthesis mechanism (MST as default)
mst_config = discrete_mechanisms.MSTConfig(seed=42)

# 4. Generate Differentially Private synthetic data
synthetic_df = dpsynth.generate(
    data=sensitive_df[['age', 'education']], # Restrict to these columns for simplicity
    domains=attribute_domains,
    epsilon=1.0,
    delta=1e-5,
    discrete_config=mst_config,
)

print("\nGenerated Synthetic Data (sample):")
print(synthetic_df.head())

# Scalable Beam API Example

This example demonstrates how to use the Scalable Beam API to generate synthetic data in a distributed manner using Apache Beam.

In [ ]:
import apache_beam as beam
from dpsynth import data_generation
from dpsynth.pipeline_transformations import types
from dpsynth import domain
from dpsynth.dataset_descriptors import csv_descriptor
import pipeline_dp
import pandas as pd

# 1. Define domain schema
# We use a small sample to create the descriptor
sample_df = pd.read_csv('adult.csv', nrows=100)
descriptor = csv_descriptor.get_dataset_descriptor_for_csv(
    dataframe=sample_df,
    field_names=["age", "education"]
)

attribute_domains = {
    'age': domain.CategoricalAttribute(possible_values=list(range(17, 91))),
    'education': domain.CategoricalAttribute(possible_values=sample_df['education'].unique().tolist())
}

# 2. Configure the synthesis mechanism
config = data_generation.DataGenerationConfig(
    epsilon=1.0,
    delta=1e-5,
    mechanism=data_generation.Mechanism.MST,
    dataset_descriptor=descriptor,
    data_format=types.DataFormat.CSV,
    num_out_records=1000,
)

# 3. Run the pipeline with BeamBackend
with beam.Pipeline() as pipeline:
    # Load data from CSV file in Beam
    # We use a simple map to parse lines for simplicity
    def csv_to_dict(line):
        values = line.split(',')
        # Assuming standard order: age is 0, education is 3
        return {'age': int(values[0]), 'education': values[3].strip()}

    raw_records = (
        pipeline
        | "ReadCSV" >> beam.io.ReadFromText('adult.csv', skip_header_lines=1)
        | "ParseCSV" >> beam.Map(csv_to_dict)
    )

    beam_backend = pipeline_dp.BeamBackend()

    synthetic_records = data_generation.generate(
        input_data=raw_records,
        config=config,
        backend=beam_backend,
    )

    # Print sample of results
    synthetic_records | "Sample" >> beam.combiners.Sample.FixedSizeGlobally(5) | "Print" >> beam.Map(print)